# CLIP Reproduction — Interactive Retrieval Demo

Given a text query, this notebook retrieves the most relevant images from Flickr30k using our trained CLIP model.

**Architecture recap:**
- Text encoder: T5-small (encoder only, ~35M params) → linear projection → 256-dim
- Image encoder: ResNet-18 pretrained → linear projection → 256-dim
- Trained with symmetric InfoNCE loss on Flickr30k

In [ ]:
import torch
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import numpy as np
from pathlib import Path
from transformers import T5Tokenizer
from datasets import load_dataset
from IPython.display import display
import ipywidgets as widgets

from model import CLIP
from demo import build_index, load_index, retrieve, show_results

%matplotlib inline
plt.rcParams['figure.dpi'] = 120

## 1. Load model and build image index

In [ ]:
CHECKPOINT = "clip_final.pt"   # <-- update if needed
INDEX_PATH = Path("image_index.pt")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

ckpt = torch.load(CHECKPOINT, map_location=device)
cfg = ckpt.get("config", {"embed_dim": 256})
model = CLIP(embed_dim=cfg["embed_dim"]).to(device)
model.load_state_dict(ckpt["model"])
model.eval()
print("Model loaded!")

tokenizer = T5Tokenizer.from_pretrained("t5-small")

In [ ]:
print("Loading Flickr30k dataset ...")
ds = load_dataset("nlphuji/flickr30k", trust_remote_code=True)["test"]
print(f"Dataset size: {len(ds)} images")

if INDEX_PATH.exists():
    print("Loading pre-built index ...")
    img_emb, captions = load_index(INDEX_PATH)
else:
    print("Building index (this takes a few minutes) ...")
    img_emb, captions, _ = build_index(model, tokenizer, device, INDEX_PATH)

print(f"Index shape: {img_emb.shape}")

## 2. Interactive text → image retrieval

In [ ]:
def query_and_show(query: str, top_k: int = 5, save_path: str = None):
    indices, scores = retrieve(query, model, tokenizer, img_emb, captions, device, top_k)
    show_results(query, indices, scores, captions, ds,
                 save_path=Path(save_path) if save_path else None)

# Try a few queries
query_and_show("a dog playing in the snow")

In [ ]:
query_and_show("people eating at a restaurant", top_k=5)

In [ ]:
query_and_show("a child riding a bicycle", top_k=5)

## 3. Widget: type any query

In [ ]:
text_input = widgets.Text(
    value='a sunset over the ocean',
    description='Query:',
    layout=widgets.Layout(width='60%')
)
k_slider = widgets.IntSlider(value=5, min=1, max=10, description='Top-k:')
button = widgets.Button(description='Search', button_style='primary')
output = widgets.Output()

def on_click(b):
    with output:
        output.clear_output(wait=True)
        query_and_show(text_input.value, top_k=k_slider.value)

button.on_click(on_click)
display(widgets.VBox([widgets.HBox([text_input, k_slider]), button, output]))

## 4. Similarity matrix visualisation (blog figure)

This is the N×N matrix from Figure 3 of the CLIP paper. The diagonal lights up as the model learns — matching image i to text i.

In [ ]:
def plot_similarity_matrix(n: int = 16, title: str = "Image–Text similarity matrix"):
    """Pick n random image-text pairs and plot their cosine similarity matrix."""
    import random
    from data import VAL_TRANSFORM

    indices = random.sample(range(len(ds)), n)
    imgs, input_ids_list, attn_masks = [], [], []

    for idx in indices:
        row = ds[idx]
        img = row["image"]
        if img.mode != "RGB":
            img = img.convert("RGB")
        imgs.append(VAL_TRANSFORM(img))

        tok = tokenizer(row["caption"][0], max_length=64,
                        padding="max_length", truncation=True, return_tensors="pt")
        input_ids_list.append(tok["input_ids"].squeeze(0))
        attn_masks.append(tok["attention_mask"].squeeze(0))

    img_tensor = torch.stack(imgs).to(device)
    ids_tensor = torch.stack(input_ids_list).to(device)
    mask_tensor = torch.stack(attn_masks).to(device)

    with torch.no_grad():
        img_emb_local = model.encode_image(img_tensor)
        txt_emb_local = model.encode_text(ids_tensor, mask_tensor)
        sim = (img_emb_local @ txt_emb_local.T * model.temperature).cpu().numpy()

    fig, ax = plt.subplots(figsize=(8, 7))
    im = ax.imshow(sim, cmap="RdBu_r")
    plt.colorbar(im, ax=ax)
    ax.set_title(title, fontsize=13, fontweight="bold")
    ax.set_xlabel("Text index", fontsize=11)
    ax.set_ylabel("Image index", fontsize=11)
    for i in range(n):
        ax.add_patch(plt.Rectangle((i - 0.5, i - 0.5), 1, 1,
                                   fill=False, edgecolor="lime", linewidth=2))
    plt.tight_layout()
    plt.savefig("similarity_matrix.png", dpi=150, bbox_inches="tight")
    plt.show()
    print("Saved → similarity_matrix.png")

plot_similarity_matrix(n=16)

## 5. Embedding space: t-SNE visualisation (blog figure)

In [ ]:
from sklearn.manifold import TSNE

def plot_tsne(n_samples: int = 500):
    """Plot image and text embeddings in 2D — pairs should cluster together."""
    import random
    from data import VAL_TRANSFORM

    indices = random.sample(range(len(ds)), n_samples)
    imgs, input_ids_list, attn_masks = [], [], []

    batch_size = 64
    all_img_emb, all_txt_emb = [], []

    for start in range(0, n_samples, batch_size):
        batch_idx = indices[start:start + batch_size]
        imgs, ids_list, masks = [], [], []
        for idx in batch_idx:
            row = ds[idx]
            img = row["image"]
            if img.mode != "RGB":
                img = img.convert("RGB")
            imgs.append(VAL_TRANSFORM(img))
            tok = tokenizer(row["caption"][0], max_length=64,
                            padding="max_length", truncation=True, return_tensors="pt")
            ids_list.append(tok["input_ids"].squeeze(0))
            masks.append(tok["attention_mask"].squeeze(0))

        with torch.no_grad():
            ie = model.encode_image(torch.stack(imgs).to(device))
            te = model.encode_text(torch.stack(ids_list).to(device),
                                   torch.stack(masks).to(device))
        all_img_emb.append(ie.cpu())
        all_txt_emb.append(te.cpu())

    img_e = torch.cat(all_img_emb).numpy()
    txt_e = torch.cat(all_txt_emb).numpy()
    combined = np.concatenate([img_e, txt_e], axis=0)

    print("Running t-SNE (may take ~30s) ...")
    tsne = TSNE(n_components=2, perplexity=30, random_state=42, n_iter=1000)
    proj = tsne.fit_transform(combined)

    img_2d = proj[:n_samples]
    txt_2d = proj[n_samples:]

    fig, ax = plt.subplots(figsize=(9, 8))
    ax.scatter(img_2d[:, 0], img_2d[:, 1], c="#3498db", alpha=0.5, s=15, label="Image")
    ax.scatter(txt_2d[:, 0], txt_2d[:, 1], c="#e74c3c", alpha=0.5, s=15, label="Text")
    # Draw lines connecting matched pairs
    for i in range(0, n_samples, 10):  # every 10th pair to keep readable
        ax.plot([img_2d[i, 0], txt_2d[i, 0]],
                [img_2d[i, 1], txt_2d[i, 1]], 'k-', alpha=0.15, linewidth=0.5)
    ax.legend(fontsize=11)
    ax.set_title("t-SNE of CLIP embeddings (image vs text)", fontsize=13, fontweight="bold")
    ax.axis("off")
    plt.tight_layout()
    plt.savefig("tsne_embeddings.png", dpi=150, bbox_inches="tight")
    plt.show()
    print("Saved → tsne_embeddings.png")

plot_tsne(n_samples=500)